# Amazon Pipeline - Full Test

Pipeline:
1. Search Amazon (curl_cffi) -> parse product cards
2. Filter on-sale + Scrape features (Approach A: specs, Approach B: product page)
3. Select top 5 deals (Cerebras via LiteLLM)
4. Estimate prices (EnsembleAgent)
5. Final results

In [1]:
# Cell 1: Setup
import os, sys, logging, time

os.chdir('/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
sys.path.insert(0, '/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
sys.path.insert(0, 'test_chuc_nang/fix_amazon_v2_s2')

logging.basicConfig(level=logging.INFO)
logging.getLogger().setLevel(logging.INFO)

from dotenv import load_dotenv
load_dotenv(override=True)

print(f'Working directory: {os.getcwd()}')
print(f'OPENROUTER_API_KEY: {"SET" if os.getenv("OPENROUTER_API_KEY") else "MISSING"}')
print('Setup complete!')

Working directory: /home/hieu0606sunny/price2026wsl/tech2ai/segment4
OPENROUTER_API_KEY: SET
Setup complete!


In [2]:
# Cell 2: Imports
import chromadb
from litellm import completion
from curl_cffi import requests as curl_requests

from price_agents.deals import Deal, DealSelection, Opportunity
from price_agents.ensemble_agent import EnsembleAgent
from buoc1_search import (
    ScrapedAmazonDeal,
    init_amazon_session,
    search_amazon,
    scrape_product_page,
    search_filter_scrape_amazon,
)
from bestbuy_untils.unified_deal import UnifiedScrapedDeal

print('Imports done!')

INFO:datasets:PyTorch version 2.9.0 available.


Imports done!


In [3]:
# Cell 3: Init EnsembleAgent (run once)
print('Initializing EnsembleAgent...')
client = chromadb.PersistentClient(path='products_vectorstore')
collection = client.get_or_create_collection('products')
print(f'ChromaDB: {collection.count()} documents')

ensemble = EnsembleAgent(collection)
print('EnsembleAgent ready!')

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Initializing EnsembleAgent...
ChromaDB: 800000 documents

INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using cuda
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready


EnsembleAgent ready!


In [ ]:
# Cell 4: Step 1+2 - Search + Filter + Scrape Amazon
TEST_KEYWORD = 'laptop gaming'

print('=' * 60)
print(f'STEP 1+2: Search + Filter + Scrape Amazon for "{TEST_KEYWORD}"')
print('=' * 60)

start = time.time()
scraped_deals = search_filter_scrape_amazon(TEST_KEYWORD, max_results=5)
elapsed = time.time() - start

print(f'\n{"="*60}')
print(f'Got {len(scraped_deals)} deals in {elapsed:.1f}s:')
for i, d in enumerate(scraped_deals, 1):
    print(f'  [{i}] ${d.price:.2f} | {d.title[:70]}')
    print(f'       Brand: {d.brand or "N/A"} | Features: {len(d.features)} chars')
    print(f'       URL: {d.url}')

STEP 1+2: Search + Filter + Scrape Amazon for "laptop gaming"


INFO:buoc1_search:[Init] ZIP 96150 set OK
INFO:buoc1_search:[Step 1] Search: https://www.amazon.com/s?k=laptop+gaming
INFO:buoc1_search:  Status: 200 | Size: 1,979,416 bytes
INFO:buoc1_search:  Found 22 products (11 on sale)
INFO:buoc1_search:[Step 2] Filtered: 11 on sale (from 22 total)
INFO:buoc1_search:
[Result] 10 deals in 2.3s (Approach A: 10, Approach B: 0)



Got 10 deals in 2.3s:
  [1] $299.98 | NIMO 15.6'' IPS FHD-Laptop, 8GB RAM 256GB SSD AMD Ryzen 5(Beat i5-1135
       Brand: N/A | Features: 85 chars
       URL: https://www.amazon.com/dp/B0DZ5LJXL3
  [2] $1799.00 | HP Omen Max 16” Gaming Laptop, AMD Ryzen AI 7 350, GeForce RTX 5070, W
       Brand: N/A | Features: 75 chars
       URL: https://www.amazon.com/dp/B0GLWRS4NL
  [3] $549.99 | NIMO 15.6" IPS FHD Gaming-Laptop, 8 Cores AMD Ryzen 7 Pro 6850U 16GB L
       Brand: N/A | Features: 90 chars
       URL: https://www.amazon.com/dp/B0GN2B8MZP
  [4] $2299.00 | ASUS ROG Strix G18 (2025) Gaming Laptop, 18” ROG Nebula 16:10 2.5K 240
       Brand: N/A | Features: 89 chars
       URL: https://www.amazon.com/dp/B0F1CB49YB
  [5] $1099.99 | ASUS TUF Gaming F16 (2025) Gaming Laptop, 16” FHD+ 165Hz 16:10 Display
       Brand: N/A | Features: 58 chars
       URL: https://www.amazon.com/dp/B0FFDDFW47
  [6] $1749.00 | HP Omen RTX 5070 AI Gaming Laptop, 17.3" FHD 144Hz (100% sRGB) HyperX 
       Bran

In [5]:
# Cell 5: Convert to UnifiedScrapedDeal (same format as BestBuy)
unified_deals = [UnifiedScrapedDeal.from_amazon(d) for d in scraped_deals]

print(f'Converted {len(unified_deals)} deals to UnifiedScrapedDeal:')
for i, ud in enumerate(unified_deals, 1):
    print(f'\n[{i}] {ud.source}: {ud.title[:70]}')
    print(f'    ${ud.price:.2f} | {ud.brand or "N/A"}')
    print(f'    Features: {ud.features[:100]}...')

Converted 10 deals to UnifiedScrapedDeal:

[1] Amazon: NIMO 15.6'' IPS FHD-Laptop, 8GB RAM 256GB SSD AMD Ryzen 5(Beat i5-1135
    $299.98 | N/A
    Features: Display Size: 15.6 inches, Disk Size: 256 GB, RAM: 8 GB, Operating System: Windows 11...

[2] Amazon: HP Omen Max 16” Gaming Laptop, AMD Ryzen AI 7 350, GeForce RTX 5070, W
    $1799.00 | N/A
    Features: Display Size: 16 inches, Disk Size: 2 TB, Operating System: Windows 11 Home...

[3] Amazon: NIMO 15.6" IPS FHD Gaming-Laptop, 8 Cores AMD Ryzen 7 Pro 6850U 16GB L
    $549.99 | N/A
    Features: Display Size: 15.60 inches, Disk Size: 512 GB, RAM: 16.00 GB, Operating System: Windows 11...

[4] Amazon: ASUS ROG Strix G18 (2025) Gaming Laptop, 18” ROG Nebula 16:10 2.5K 240
    $2299.00 | N/A
    Features: Display Size: 18 inches, Disk Size: 2000 GB, RAM: 32 GB, Operating System: Windows 11 Pro...

[5] Amazon: ASUS TUF Gaming F16 (2025) Gaming Laptop, 16” FHD+ 165Hz 16:10 Display
    $1099.99 | N/A
    Features: Display Size: 16 inc

In [ ]:
from openai import OpenAI                                                                                                                                      
                                                                                                                                                                 
print('=' * 60)                                                                                                                                                
print('STEP 3: Select top 3 deals (GPT-5-mini)')                                                                                                               
print('=' * 60)                                                                                                                                                
                                                                                                                                                                
SYSTEM_PROMPT = """You identify and summarize the 3 most detailed deals from a list, by selecting deals that have the most detailed, high quality description  
and the most clear price.                                                                                                                                      
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description.                        
Most important is that you respond with the 3 deals that have the most detailed product description with price.                                                
                                                                                                                                                                
**IMPORTANT:**                                                                                                                                                 
1. Focus on the product features and specifications, not sales terms.                                                                                          
2. The product_description should be a 3-4 sentence summary of the product itself.                                                                             
3. Price must be greater than 0.                                                                                                                               
4. Keep the original URL exactly as provided."""                                                                                                               
                                                                                                                                                                
USER_PROMPT_PREFIX = """Respond with the most promising 3 deals from this list, selecting those which have the most detailed, high quality product description 
and a clear price that is greater than 0.                                                                                                                      
You should rephrase the description to be a summary of the product itself, not the terms of the deal.                                                          
Remember to respond with a short paragraph of text in the product_description field for each of the 3 items that you select.                                   
                                                                                                                                                                
Deals:                                                                                                                                                         
                                                                                                                                                                
"""                                                                                                                                                         

valid_deals = [ud for ud in unified_deals if ud.price > 0]
user_prompt = USER_PROMPT_PREFIX                                                                                                                               
user_prompt += '\n\n'.join([ud.describe() for ud in valid_deals])                                                                                              
user_prompt += '\n\nInclude up to 5 deals, no more.'                                                                                                           
                                                                                                                                                                
start = time.time()                                                                                                                                            
print(f'Calling GPT-5-mini with {len(valid_deals)} deals...')                                                                                                  
                                                                                                                                                            
openai_client = OpenAI()
result = openai_client.chat.completions.parse(                                                                                                                 
    model="gpt-5-mini",                                                                                                                                        
    messages=[                                                                                                                                                 
        {"role": "system", "content": SYSTEM_PROMPT},                                                                                                          
        {"role": "user", "content": user_prompt},                                                                                                           
    ],                                                                                                                                                         
    response_format=DealSelection,                                                                                                                             
    reasoning_effort="minimal",                                                                                                                                
)                                                                                                                                                              
                                                                                                                                                            
deal_selection = result.choices[0].message.parsed
deal_selection.deals = [d for d in deal_selection.deals if d.price > 0]                                                                                        
                                                                                                                                                                
elapsed = time.time() - start                                                                                                                                  
print(f'\nSelected {len(deal_selection.deals)} deals in {elapsed:.1f}s:')                                                                                      
for i, deal in enumerate(deal_selection.deals, 1):                                                                                                             
    print(f'  [{i}] ${deal.price:.2f} | {deal.product_description[:80]}...')                                                                                   
    print(f'       URL: {deal.url}')

STEP 3: Select top 5 deals (GPT-5-mini)
Calling GPT-5-mini with 10 deals...

Selected 5 deals in 12.8s:
  [1] $299.98 | 15.6" IPS Full HD laptop powered by an AMD Ryzen 5 processor with up to 3.7GHz b...
       URL: https://www.amazon.com/dp/B0DZ5LJXL3
  [2] $1799.00 | 16" gaming laptop featuring the latest AMD Ryzen AI 7 350 CPU combined with an N...
       URL: https://www.amazon.com/dp/B0GLWRS4NL
  [3] $2299.00 | 18" ROG Nebula 2.5K (16:10) display with 240Hz refresh and 3ms response designed...
       URL: https://www.amazon.com/dp/B0F1CB49YB
  [4] $1099.99 | 16" FHD+ 165Hz 16:10 gaming laptop driven by an Intel Core i5-13450HX CPU and NV...
       URL: https://www.amazon.com/dp/B0FFDDFW47
  [5] $599.99 | 15.6" IPS Full HD laptop built around an AMD R7 7735HS (8 cores/16 threads, up t...
       URL: https://www.amazon.com/dp/B0GN2YNVJR


In [8]:
# Cell 7: Step 4 - Estimate prices (EnsembleAgent)
print('=' * 60)
print('STEP 4: Estimate prices with EnsembleAgent')
print('=' * 60)

start = time.time()
opportunities = []

for i, deal in enumerate(deal_selection.deals, 1):
    print(f'\n[{i}/{len(deal_selection.deals)}] {deal.product_description[:50]}...')
    estimate = ensemble.price(deal.product_description)
    discount = estimate - deal.price
    opportunities.append(Opportunity(deal=deal, estimate=estimate, discount=discount))
    print(f'  Sale: ${deal.price:.2f} | Est: ${estimate:.2f} | Discount: ${discount:.2f}')

print(f'\nEstimation done in {time.time() - start:.1f}s')

INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
11:45:57 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


STEP 4: Estimate prices with EnsembleAgent

[1/5] 15.6" IPS Full HD laptop powered by an AMD Ryzen 5...


11:45:57 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $350.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $429.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $555.71
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $433.77
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
11:46:49 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


...

  Sale: $1099.99 | Est: $1017.71 | Discount: $-82.28

[5/5] 15.6" IPS Full HD laptop built around an AMD R7 77...


11:46:59 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $700.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $749.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $740.31
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $743.23


  Sale: $599.99 | Est: $743.23 | Discount: $143.24

Estimation done in 65.0s


In [9]:
# Cell 8: Final Results
print('=' * 80)
print('FINAL RESULTS - Amazon Deals Sorted by Discount')
print('=' * 80)
print(f'Keyword: "{TEST_KEYWORD}"\n')

opportunities.sort(key=lambda x: x.discount, reverse=True)

for i, opp in enumerate(opportunities, 1):
    pct = (opp.discount / opp.estimate * 100) if opp.estimate > 0 else 0
    status = 'HOT DEAL' if opp.discount > 200 else 'Good Deal' if opp.discount > 100 else 'OK' if opp.discount > 0 else 'Overpriced'
    
    print(f'--- #{i} [{status}] ---')
    print(f'  Product:  {opp.deal.product_description[:80]}...')
    print(f'  Sale:     ${opp.deal.price:.2f}')
    print(f'  Estimate: ${opp.estimate:.2f}')
    print(f'  Discount: ${opp.discount:.2f} ({pct:.0f}%)')
    print(f'  URL:      {opp.deal.url}')
    print()

FINAL RESULTS - Amazon Deals Sorted by Discount
Keyword: "laptop gaming"

--- #1 [Good Deal] ---
  Product:  15.6" IPS Full HD laptop built around an AMD R7 7735HS (8 cores/16 threads, up t...
  Sale:     $599.99
  Estimate: $743.23
  Discount: $143.24 (19%)
  URL:      https://www.amazon.com/dp/B0GN2YNVJR

--- #2 [Good Deal] ---
  Product:  15.6" IPS Full HD laptop powered by an AMD Ryzen 5 processor with up to 3.7GHz b...
  Sale:     $299.98
  Estimate: $433.77
  Discount: $133.79 (31%)
  URL:      https://www.amazon.com/dp/B0DZ5LJXL3

--- #3 [Overpriced] ---
  Product:  16" FHD+ 165Hz 16:10 gaming laptop driven by an Intel Core i5-13450HX CPU and NV...
  Sale:     $1099.99
  Estimate: $1017.71
  Discount: $-82.28 (-8%)
  URL:      https://www.amazon.com/dp/B0FFDDFW47

--- #4 [Overpriced] ---
  Product:  16" gaming laptop featuring the latest AMD Ryzen AI 7 350 CPU combined with an N...
  Sale:     $1799.00
  Estimate: $1613.92
  Discount: $-185.08 (-11%)
  URL:      https://www.amaz